In [2]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from lightgbm import LGBMClassifier

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "outputs" / "APL_Logistics_ml_ready.csv"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project root:",PROJECT_ROOT)
print("Data path:",DATA_PATH)
print("Model directory:",MODEL_DIR)
print("Output directory:",OUTPUT_DIR)

Project root: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project
Data path: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\outputs\APL_Logistics_ml_ready.csv
Model directory: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\models
Output directory: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\outputs


In [4]:
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Shape:",df.shape)
df.head()

Shape: (180519, 37)


,Type,Days for shipment (scheduled),Benefit per order,Sales per customer,Category Id,Category Name,Customer City,Customer Country,Customer Segment,Customer State,...,Product Name,Product Price,Shipping Mode,discount_amount_per_unit,sales_per_quantity,profit_per_quantity,price_discount_interaction,scheduled_days_bucket,geo_market_region,Late_delivery_risk
0,DEBIT,4,159.69,472.45,9,Cardio Equipment,Brownsville,EE. UU.,Consumer,TX,...,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class,5.500,99.99,31.938,5.9994,4,Pacific Asia | South Asia,1
1,DEBIT,4,48.71,167.96,29,Shop By Sport,Littleton,EE. UU.,Consumer,CO,...,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class,6.398,39.99,9.742,6.3984,4,LATAM | Central America,0
2,DEBIT,4,87.36,181.99,48,Water Sports,Littleton,EE. UU.,Consumer,CO,...,Pelican Sunstream 100 Kayak,199.99,Standard Class,18.000,199.99,87.360,17.9991,4,LATAM | Central America,0
3,DEBIT,4,-41.89,175.99,48,Water Sports,Littleton,EE. UU.,Consumer,CO,...,Pelican Sunstream 100 Kayak,199.99,Standard Class,24.000,199.99,-41.890,23.9988,4,USCA | East of USA,1
4,DEBIT,4,10.00,40.00,24,Women's Apparel,Littleton,EE. UU.,Consumer,CO,...,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,10.000,50.00,10.000,10.0000,4,USCA | East of USA,1


In [5]:
TARGET = "Late_delivery_risk"

print(df[TARGET].value_counts())
print()
print(df[TARGET].value_counts(normalize=True))

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Late_delivery_risk
1    0.548291
0    0.451709
Name: proportion, dtype: float64


In [6]:
x = df.drop(columns=[TARGET].copy())
y = df[TARGET].astype(int).copy()

print("x shape:",x.shape)
print("y shape:",y.shape)

x shape: (180519, 36)
y shape: (180519,)


In [7]:
LEAKAGE_COLUMNS = [
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Delivery Status",
    "Order Status",
    "Delay_Gap"
]

leakage_present = [
    col for col in LEAKAGE_COLUMNS
    if col in x.columns
]

print("Leakage columns still present:")
print(leakage_present)

Leakage columns still present:
[]


In [8]:
TIMING_SENSITIVE_COLUMNS = [
    "Order Profit Per Order",
    "profit_per_quantity",
    "Benefit per order"
]

present_timing_sensitive = [
    col for col in TIMING_SENSITIVE_COLUMNS
    if col in x.columns
]

print("Timing-sensitive columns found:")
print(present_timing_sensitive)

Timing-sensitive columns found:
['Order Profit Per Order', 'profit_per_quantity', 'Benefit per order']


In [9]:
x= x.drop(
    columns = present_timing_sensitive,
    errors= "ignore"
)

print("New x shape:",x.shape)

New x shape: (180519, 33)


In [11]:
numeric_features = x.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = x.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical features:",len(numeric_features))
print("Numerical features:")
print(numeric_features)

print()

print("Number of categorical features:",len(categorical_features))
print("Categorical features:")
print(categorical_features)

Number of numerical features: 18
Numerical features:
['Days for shipment (scheduled)', 'Sales per customer', 'Category Id', 'Department Id', 'Latitude', 'Longitude', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Product Price', 'discount_amount_per_unit', 'sales_per_quantity', 'price_discount_interaction', 'scheduled_days_bucket']

Number of categorical features: 15
Categorical features:
['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Product Name', 'Shipping Mode', 'geo_market_region']


C:\Users\ganesh\AppData\Local\Temp\ipykernel_3136\1658122694.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x.select_dtypes(
